# Visual Asset Auditing System - Test Bench

This notebook implements the **Test Bench** tier of the Visual Asset Auditing System. It demonstrates:

1. Connecting to AlloyDB and GCS.
2. Running a Hybrid Search (pgvector + FTS).
3. Detecting the drop-off point and selecting the top 60 candidates (High Confidence + Borderline).
4. Running Gemini 3.5 Flash online inference to audit the selected assets.
5. Creating and updating the `audit_results` table in AlloyDB.

*Transcribed from IMG_9422.jpeg. Additional source screenshots will be added below in the order received.*

In [ ]:
# Install required libraries
!pip install -q google-genai google-cloud-vision google-cloud-aiplatform kfp google-cloud-pipeline-components pgvector asyncpg kneed pandas numpy pillow nest-asyncio sqlalchemy "protobuf<5.0.0dev"


## Package Installation

This cell installs all necessary external libraries (Google GenAI, Cloud Vision, AI Platform, pgvector, asyncio) to equip the notebook environment.

In [ ]:
# Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()

import google.genai as genai
from google.genai import types

print("Google GenAI SDK imported successfully.")


## Google Cloud Authentication

This cell handles authentication with Google Cloud using Colab auth utilities, sets up API project client.

In [ ]:
# === CONFIGURATION CONSTANTS ===
# Edit this cell to change models, thresholds, or infrastructure settings.
# All downstream cells read from these constants — never from hardcoded values.
import math, io
from typing import Literal, Optional, List, Dict, Tuple

# --- Model Configuration ---
GEMINI_ORCHESTRATOR_MODEL = "gemini-2.5-pro"       # Forensic analysis + audit config (needs reasoning)
GEMINI_INFERENCE_MODEL    = "gemini-2.5-flash"     # Per-image visual audit (speed + cost)
# GEMINI_CROSS_ENCODER_MODEL: REMOVED — replaced by zero-latency CPU reranker (no LLM calls at stage 3.5)
EMBEDDING_MODEL           = "multimodalembedding@001"  # Vertex AI multimodal embedding

# --- Infrastructure (fill in your values) ---
PROJECT_ID       = "YOUR_PROJECT_ID"
REGION           = "YOUR_REGION"
ALLOYDB_CLUSTER  = "YOUR_CLUSTER_ID"
ALLOYDB_INSTANCE = "YOUR_INSTANCE_ID"
DB_USER          = "YOUR_DB_USER"
DB_PASSWORD      = "YOUR_DB_PASSWORD"
DB_NAME          = "YOUR_DB_NAME"
DB_SCHEMA        = "public"

# --- Initialize GenAI client ---
import google.genai as genai
from google.genai import types
client = genai.Client(project=PROJECT_ID, location=REGION, vertexai=True)

# ─── Search Mode Parameters ─────────────────────────────────────────────────
# Vision tag filter is LLM-chosen at Stage 1 from the top-N most frequent DB tags.
# It scopes the tag-search arm to only images Cloud Vision labelled as relevant,
# and acts as a multiplier in the CPU reranker.
# These dicts tune behaviour per mode — all values calibrated for
# high accuracy + high precision + high recall simultaneously.

# Vector cosine distance below which a candidate is force-rescued from Low → Edge tier
VECTOR_SAFEGUARD_THRESHOLD = {
    "LOGO_SIMILARITY": 0.28,   # Tight — logos must be visually close
    "UI_COMPONENT":    0.32,   # Slightly looser — UI variants differ more
    "PERSON_SEARCH":   0.50,   # Generous — face in corner of a screenshot has high distance
    "EXACT_MATCH":     0.10,   # Near-pixel-perfect only
    "GENERAL_ASSET":   0.35,
}

# Kneedle sensitivity: higher = sharper cutoff (precision), lower = softer (recall)
KNEEDLE_SENSITIVITY = {
    "LOGO_SIMILARITY": 1.2,
    "UI_COMPONENT":    1.0,
    "PERSON_SEARCH":   0.7,    # Soft — keep more borderline candidates
    "EXACT_MATCH":     1.5,    # Very sharp — near-exact only
    "GENERAL_ASSET":   1.0,
}

# LLM match_confidence below which a PASS verdict is demoted to FAIL
CONFIDENCE_THRESHOLD = {
    "LOGO_SIMILARITY": 72,
    "UI_COMPONENT":    70,
    "PERSON_SEARCH":   50,     # Any plausible face match should surface
    "EXACT_MATCH":     88,     # Near-certain required for exact match
    "GENERAL_ASSET":   68,
}

# RRF fusion weights per arm (must sum to 1.0 per mode)
RRF_WEIGHTS = {
    "LOGO_SIMILARITY": {"vector": 0.65, "fts": 0.20, "tags": 0.15, "face": 0.00},
    "UI_COMPONENT":    {"vector": 0.60, "fts": 0.25, "tags": 0.15, "face": 0.00},
    "PERSON_SEARCH":   {"vector": 0.45, "fts": 0.10, "tags": 0.15, "face": 0.30},
    "EXACT_MATCH":     {"vector": 0.80, "fts": 0.10, "tags": 0.10, "face": 0.00},
    "GENERAL_ASSET":   {"vector": 0.60, "fts": 0.25, "tags": 0.15, "face": 0.00},
}

# Contrastive negative vector weight (0.0 = pure HNSW, no negative subtraction)
# Logo modes use contrastive to suppress wrong-brand lookalikes.
# Person/Exact use pure ANN for maximum recall.
NEGATIVE_VECTOR_WEIGHT = {
    "LOGO_SIMILARITY": 0.30,
    "UI_COMPONENT":    0.20,
    "PERSON_SEARCH":   0.00,
    "EXACT_MATCH":     0.00,
    "GENERAL_ASSET":   0.15,
}

# ─── Person Search — Cloud Vision face/person tag vocabulary ────────────────
# Used ONLY for the 4th "face arm" in PERSON_SEARCH mode.
# The LLM-chosen vision_tag_filter (from audit config) scopes Arm 3 separately.
PERSON_SEARCH_FACE_TAGS = [
    "Person", "People", "Face", "Head", "Human", "Human body",
    "Portrait", "Selfie", "Smile", "Facial expression", "Nose", "Eyebrow",
    "Adult", "Man", "Woman", "Child", "Boy", "Girl",
    "Forehead", "Cheek", "Chin", "Hair", "Beard", "Hairstyle",
    "Conversation", "Meeting", "Presentation", "Profile picture",
]

# ─── Static tag fallback ────────────────────────────────────────────────────
# Used when DB tag query fails. Also serves as reference vocabulary
# for what kinds of tags Cloud Vision generates (passed to LLM).
WEB_AUDIT_VISION_TAGS = [
    # Logos & Branding
    "Logo", "Brand", "Trademark", "Wordmark", "Emblem", "Badge", "Seal",
    # UI Components
    "Button", "Icon", "Screenshot", "Web page", "User interface", "Mobile phone",
    "Application software", "Font", "Text", "Signage", "Banner", "Toolbar",
    "Modal", "Dialog", "Popup", "Navigation", "Menu", "Form", "Input",
    # Shapes & Design
    "Circle", "Rectangle", "Square", "Triangle", "Symbol", "Pattern",
    "Graphic design", "Illustration", "Clip art", "Drawing", "Animation",
    # People
    "Person", "People", "Face", "Portrait", "Selfie", "Head", "Human",
    # Media
    "Image", "Photo", "Photograph", "Thumbnail", "Preview",
    # Colors (for color-sensitive audits)
    "Red", "Blue", "Green", "Yellow", "Orange", "Purple", "Black", "White",
    # Payment & finance (common audit domain)
    "Credit card", "Payment", "QR code", "Barcode",
    # Notification / feedback UI
    "Notification", "Alert", "Checkbox", "Toggle", "Radio button",
]

# Max tags passed to LLM from the DB frequency query (prevents 70k → 429 overflow)
MAX_TAGS_TO_LLM = 300

# Inference concurrency (Gemini Flash handles high parallel throughput well)
MAX_INFERENCE_WORKERS = 20

# Candidate caps going into LLM inference
MAX_CANDIDATES_QUICK_MODE = 25
MAX_CANDIDATES_SMART_SCAN = 60

print("✅ Configuration constants loaded.")
print(f"   Orchestrator : {GEMINI_ORCHESTRATOR_MODEL}")
print(f"   Inference    : {GEMINI_INFERENCE_MODEL}")
print(f"   Embedding    : {EMBEDDING_MODEL}")
print(f"   Static tags  : {len(WEB_AUDIT_VISION_TAGS)} (fallback)")
print(f"   Face tags    : {len(PERSON_SEARCH_FACE_TAGS)} (person arm)")


In [ ]:
# === CONFIGURATION CONSTANTS ===
# Edit this cell to change models, thresholds, or infrastructure settings.
# All downstream cells read from these constants — never from hardcoded values.
import math, io
from typing import Literal, Optional, List, Dict, Tuple

# --- Model Configuration ---
GEMINI_ORCHESTRATOR_MODEL = "gemini-2.5-pro"       # Forensic analysis + audit config (needs reasoning)
GEMINI_INFERENCE_MODEL    = "gemini-2.5-flash"     # Per-image visual audit (speed + cost)
# GEMINI_CROSS_ENCODER_MODEL: REMOVED — replaced by zero-latency CPU reranker (no LLM calls at stage 3.5)
EMBEDDING_MODEL           = "multimodalembedding@001"  # Vertex AI multimodal embedding

# --- Infrastructure (fill in your values) ---
PROJECT_ID       = "YOUR_PROJECT_ID"
REGION           = "YOUR_REGION"
ALLOYDB_CLUSTER  = "YOUR_CLUSTER_ID"
ALLOYDB_INSTANCE = "YOUR_INSTANCE_ID"
DB_USER          = "YOUR_DB_USER"
DB_PASSWORD      = "YOUR_DB_PASSWORD"
DB_NAME          = "YOUR_DB_NAME"
DB_SCHEMA        = "public"

# --- Initialize GenAI client ---
import google.genai as genai
from google.genai import types
client = genai.Client(project=PROJECT_ID, location=REGION, vertexai=True)

# ─── Search Mode Parameters ─────────────────────────────────────────────────
# Vision tag filter is LLM-chosen at Stage 1 from the top-N most frequent DB tags.
# It scopes the tag-search arm to only images Cloud Vision labelled as relevant,
# and acts as a multiplier in the CPU reranker.
# These dicts tune behaviour per mode — all values calibrated for
# high accuracy + high precision + high recall simultaneously.

# Vector cosine distance below which a candidate is force-rescued from Low → Edge tier
VECTOR_SAFEGUARD_THRESHOLD = {
    "LOGO_SIMILARITY": 0.28,   # Tight — logos must be visually close
    "UI_COMPONENT":    0.32,   # Slightly looser — UI variants differ more
    "PERSON_SEARCH":   0.50,   # Generous — face in corner of a screenshot has high distance
    "EXACT_MATCH":     0.10,   # Near-pixel-perfect only
    "GENERAL_ASSET":   0.35,
}

# Kneedle sensitivity: higher = sharper cutoff (precision), lower = softer (recall)
KNEEDLE_SENSITIVITY = {
    "LOGO_SIMILARITY": 1.2,
    "UI_COMPONENT":    1.0,
    "PERSON_SEARCH":   0.7,    # Soft — keep more borderline candidates
    "EXACT_MATCH":     1.5,    # Very sharp — near-exact only
    "GENERAL_ASSET":   1.0,
}

# LLM match_confidence below which a PASS verdict is demoted to FAIL
CONFIDENCE_THRESHOLD = {
    "LOGO_SIMILARITY": 72,
    "UI_COMPONENT":    70,
    "PERSON_SEARCH":   50,     # Any plausible face match should surface
    "EXACT_MATCH":     88,     # Near-certain required for exact match
    "GENERAL_ASSET":   68,
}

# RRF fusion weights per arm (must sum to 1.0 per mode)
RRF_WEIGHTS = {
    "LOGO_SIMILARITY": {"vector": 0.65, "fts": 0.20, "tags": 0.15, "face": 0.00},
    "UI_COMPONENT":    {"vector": 0.60, "fts": 0.25, "tags": 0.15, "face": 0.00},
    "PERSON_SEARCH":   {"vector": 0.45, "fts": 0.10, "tags": 0.15, "face": 0.30},
    "EXACT_MATCH":     {"vector": 0.80, "fts": 0.10, "tags": 0.10, "face": 0.00},
    "GENERAL_ASSET":   {"vector": 0.60, "fts": 0.25, "tags": 0.15, "face": 0.00},
}

# Contrastive negative vector weight (0.0 = pure HNSW, no negative subtraction)
# Logo modes use contrastive to suppress wrong-brand lookalikes.
# Person/Exact use pure ANN for maximum recall.
NEGATIVE_VECTOR_WEIGHT = {
    "LOGO_SIMILARITY": 0.30,
    "UI_COMPONENT":    0.20,
    "PERSON_SEARCH":   0.00,
    "EXACT_MATCH":     0.00,
    "GENERAL_ASSET":   0.15,
}

# ─── Person Search — Cloud Vision face/person tag vocabulary ────────────────
# Used ONLY for the 4th "face arm" in PERSON_SEARCH mode.
# The LLM-chosen vision_tag_filter (from audit config) scopes Arm 3 separately.
PERSON_SEARCH_FACE_TAGS = [
    "Person", "People", "Face", "Head", "Human", "Human body",
    "Portrait", "Selfie", "Smile", "Facial expression", "Nose", "Eyebrow",
    "Adult", "Man", "Woman", "Child", "Boy", "Girl",
    "Forehead", "Cheek", "Chin", "Hair", "Beard", "Hairstyle",
    "Conversation", "Meeting", "Presentation", "Profile picture",
]

# ─── Static tag fallback ────────────────────────────────────────────────────
# Used when DB tag query fails. Also serves as reference vocabulary
# for what kinds of tags Cloud Vision generates (passed to LLM).
WEB_AUDIT_VISION_TAGS = [
    # Logos & Branding
    "Logo", "Brand", "Trademark", "Wordmark", "Emblem", "Badge", "Seal",
    # UI Components
    "Button", "Icon", "Screenshot", "Web page", "User interface", "Mobile phone",
    "Application software", "Font", "Text", "Signage", "Banner", "Toolbar",
    "Modal", "Dialog", "Popup", "Navigation", "Menu", "Form", "Input",
    # Shapes & Design
    "Circle", "Rectangle", "Square", "Triangle", "Symbol", "Pattern",
    "Graphic design", "Illustration", "Clip art", "Drawing", "Animation",
    # People
    "Person", "People", "Face", "Portrait", "Selfie", "Head", "Human",
    # Media
    "Image", "Photo", "Photograph", "Thumbnail", "Preview",
    # Colors (for color-sensitive audits)
    "Red", "Blue", "Green", "Yellow", "Orange", "Purple", "Black", "White",
    # Payment & finance (common audit domain)
    "Credit card", "Payment", "QR code", "Barcode",
    # Notification / feedback UI
    "Notification", "Alert", "Checkbox", "Toggle", "Radio button",
]

# Max tags passed to LLM from the DB frequency query (prevents 70k → 429 overflow)
MAX_TAGS_TO_LLM = 300

# Inference concurrency (Gemini Flash handles high parallel throughput well)
MAX_INFERENCE_WORKERS = 20

# Candidate caps going into LLM inference
MAX_CANDIDATES_QUICK_MODE = 25
MAX_CANDIDATES_SMART_SCAN = 60

print("✅ Configuration constants loaded.")
print(f"   Orchestrator : {GEMINI_ORCHESTRATOR_MODEL}")
print(f"   Inference    : {GEMINI_INFERENCE_MODEL}")
print(f"   Embedding    : {EMBEDDING_MODEL}")
print(f"   Static tags  : {len(WEB_AUDIT_VISION_TAGS)} (fallback)")
print(f"   Face tags    : {len(PERSON_SEARCH_FACE_TAGS)} (person arm)")


In [ ]:
# AlloyDB Connection Setup using SQLAlchemy Pool + AsyncConnector
import asyncio
import asyncpg
from typing import Tuple
from sqlalchemy.ext.asyncio import create_async_engine, AsyncEngine
from google.cloud.alloydb.connector import IPTypes, AsyncConnector

_engine_cache = {}
_connector_cache = {}

async def get_alloydb_connection(reuse: bool = True) -> Tuple[AsyncEngine, AsyncConnector]:
    """Establishes and pools AlloyDB connections using SQLAlchemy and the AsyncConnector, with automatic timeout diagnostics."""
    global _engine_cache
    global _connector_cache

    if reuse and 'default' in _engine_cache:
        return _engine_cache['default'], _connector_cache['default']

    # Use lazy refresh for serverless/Colab environments
    connector = AsyncConnector(refresh_strategy="lazy")

    async def getconn():
        # Handle case where user pasted the full resource path or just the instance ID
        instance_uri = ALLOYDB_INSTANCE
        if not instance_uri.startswith("projects/"):
            instance_uri = f"projects/{PROJECT_ID}/locations/{REGION}/clusters/{ALLOYDB_CLUSTER}/instances/{ALLOYDB_INSTANCE}"

        try:
            # Enforce 10-second connection timeout to prevent hanging loop CancelledErrors
            conn = await asyncio.wait_for(
                connector.connect(
                    instance_uri,
                    "asyncpg",
                    user=DB_USER,
                    password=DB_PASSWORD,
                    db=DB_NAME,
                    enable_iam_auth=False, # Set to True if using IAM auth
                    ip_type=IPTypes.PUBLIC # Adjust to PUBLIC or PRIVATE
                ),
                timeout=10.0
            )
            return conn
        except asyncio.TimeoutError:
            raise ConnectionError(
                f"AlloyDB connection timed out (10s) to {instance_uri}. "
                "Ensure your client IP is authorized in the AlloyDB Public IP console, "
                "or check your VPC network access if running internally."
            )
        except Exception as e:
            raise ConnectionError(f"Failed to connect to AlloyDB: {e}")

    engine = create_async_engine(
        "postgresql+asyncpg://",
        async_creator=getconn,
        echo=False,
        pool_size=10,
        max_overflow=20,
        pool_pre_ping=True # Force SQLAlchemy to health-check connections
    )

    if reuse:
        _engine_cache['default'] = engine
        _connector_cache['default'] = connector

    return engine, connector


In [ ]:
# # Run the setup
# import nest_asyncio
# nest_asyncio.apply()
# try:
#     asyncio.run(setup_database())
# except Exception as e:
#     print(f"Skipping execution: Database connection not configured yet ({e})")


In [ ]:
# Database Connection Verification (Read-Only Check – No DDL or Schema Changes)
import asyncio
import nest_asyncio

async def verify_database_connection():
    """Verifies connection to AlloyDB and confirms the visual_assets table is reachable without making any DDL changes."""
    try:
        engine, _ = await get_alloydb_connection()
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            count = await db.fetchval(f"SELECT COUNT(*) FROM {DB_SCHEMA}.visual_assets;")
            print(f"Connected to AlloyDB successfully. Schema '{DB_SCHEMA}.visual_assets' is ready ({count} assets found).")
    except Exception as e:
        print(f"Database connection status: {e}")

# Run connection check
nest_asyncio.apply()
try:
    asyncio.run(verify_database_connection())
except Exception as e:
    print(f"Skipping execution: Database connection not configured yet ({e})")


In [ ]:
# 1. Audit Config Generator & Embedding Generation
# vision_tag_filter = LLM-chosen subset of the top-300 most frequent DB tags.
# These tags scope the tag-search arm (Arm 3) and act as a bounded boost
# in the CPU reranker — making both search and reranking more precise.
# The 70k SELECT DISTINCT overflow fix is in generate_audit_config_only (Stage 1).
#
# OPTIMIZATIONS (same structure, same names):
#   - Forensic analysis + config generation are now ONE Gemini Pro call
#     (generate_audit_config accepts the reference image directly) — halves
#     Stage-1 latency and cost. image_description is filled by the same call.
#   - adjudication_logic upgraded to a 5-step human-auditor protocol
#     (LOCATE -> SCAN -> COMPARE -> EXCLUDE -> VERDICT) that always compares
#     IMAGE 0 (target) against IMAGE 1 (candidate) explicitly.
#   - Embedding: when a reference image exists, the IMAGE ALONE is embedded.
#     Corpus vectors are image embeddings — blending query prose into the image
#     call shifts the query vector away from every DB row (recall loss).
#     Caption-style text embedding is used only for text-to-image + the negative.
import json
import asyncio
from concurrent.futures import ThreadPoolExecutor
from typing import Optional, List, Dict, Tuple
from pydantic import BaseModel, Field


class AuditContextModel(BaseModel):
    search_mode: Literal["LOGO_SIMILARITY", "UI_COMPONENT", "PERSON_SEARCH", "EXACT_MATCH", "GENERAL_ASSET"] = Field(
        description=(
            "Primary search mode detected from the user goal and reference image. "
            "LOGO_SIMILARITY: brand logos/icons. UI_COMPONENT: buttons/modals/banners/UI elements. "
            "PERSON_SEARCH: finding a specific person by face (even if tiny/occluded). "
            "EXACT_MATCH: near-identical reverse image search. GENERAL_ASSET: anything else."
        )
    )
    audit_goal: str = Field(description="Refined, precise version of the user's audit goal.")
    image_description: Optional[str] = Field(
        None,
        description=(
            "Forensic description of the reference image if one is attached, else null. "
            "Must cover: content type; brand/subject identity (name the brand and the SPECIFIC "
            "version/generation if identifiable); exact visual signatures — shapes and outlines, "
            "typography (exact spelling, case, weight), sub-element layout, colours/gradients, "
            "design generation (legacy vs modern, flat vs 3D, filled vs outlined). "
            "For a person: facial bone structure only (face shape, eye spacing, nose, jawline, "
            "permanent distinctive features) — never clothing, hair colour, or background."
        )
    )
    inclusion_criteria: List[str] = Field(
        description="3-6 specific, visually testable criteria an image MUST meet to be relevant."
    )
    exclusion_criteria: List[str] = Field(
        description="2-4 criteria that EXCLUDE an image. Include lookalike/anti-spoofing rules."
    )
    adjudication_logic: str = Field(
        description=(
            "Numbered 5-step decision protocol the inference LLM will execute — NOT prose. "
            "Written the way a careful human reviewer inspects an image. Format exactly: "
            "'STEP 1 (LOCATE): in IMAGE 0 identify the target element and its defining features. "
            "STEP 2 (SCAN): sweep IMAGE 1 region by region, including inside screenshots, banners "
            "and thumbnails — the target may be tiny, cropped or partially occluded. "
            "STEP 3 (COMPARE): place the located element beside IMAGE 0 and check feature by feature. "
            "STEP 4 (EXCLUDE): rule out lookalikes. "
            "STEP 5 (VERDICT): PASS only if [exact condition]; FAIL otherwise.' "
            "Must be specific enough that two different auditors following it would reach the same verdict."
        )
    )
    search_keywords: List[str] = Field(
        description=(
            "5-8 distinctive single-word terms for full-text search against Gemini-generated image "
            "descriptions in the database. Terms are OR-combined, so every word must be discriminative "
            "on its own (brand names, element names — never generic words like 'image' or 'blue'). "
            "Single words only - no phrases."
        )
    )
    vision_tag_filter: List[str] = Field(
        description=(
            "2-5 tags chosen STRICTLY from the provided available_tags list. "
            "These scope the tag-search arm to only images Cloud Vision labelled as relevant, "
            "and boost scores in the CPU reranker. Pick the tags most specific to the audit target. "
            "DO NOT invent tags not present in the list."
        )
    )
    audit_instructions: str = Field(
        description="Concise guidance for the visual auditing LLM. Max 120 words. No repetition of criteria."
    )
    extraction_schema: Dict[str, str] = Field(
        description=(
            "Additional fields to extract per image. Values = field type + description. "
            "Always include: detected_asset_style (string), is_outdated_or_noncompliant (boolean), "
            "is_embedded_in_composite (boolean), optical_resolution_sufficient (boolean)."
        )
    )


def get_image_mime_type(path: str) -> str:
    lower = path.lower()
    if lower.endswith(".png"):  return "image/png"
    if lower.endswith(".webp"): return "image/webp"
    if lower.endswith(".gif"):  return "image/gif"
    return "image/jpeg"


def generate_audit_config(user_goal: str,
                           reference_image_description: Optional[str] = None,
                           available_tags: Optional[List[str]] = None,
                           reference_image_part=None) -> dict:
    """
    Translates a user goal into a structured, mode-aware audit configuration.
    ONE Gemini Pro call: when reference_image_part is given, the model performs the
    forensic analysis (-> image_description) and the config generation together —
    no separate forensic round-trip.
    The LLM selects vision_tag_filter from available_tags (top-300 by frequency from DB)
    to scope the tag search arm and CPU reranker to relevant images only.
    """
    image_context = (
        f"\nReference Image Forensic Analysis:\n{reference_image_description}\n"
        if reference_image_description else ""
    )
    active_tags = available_tags if available_tags else WEB_AUDIT_VISION_TAGS
    tags_str = ", ".join([f"'{t}'" for t in active_tags])

    prompt = (
        "You are an expert visual asset auditor. Translate the user's audit goal into a precise, "
        "structured audit configuration for a downstream visual-comparison LLM.\n\n"
        f"User Goal: {user_goal}\n"
        f"{image_context}\n"
        "STEP 0 - FORENSIC ANALYSIS (only if a reference image is attached to this request)\n"
        "Examine the attached reference image like a forensic analyst and write image_description:\n"
        "- Content type: standalone logo/icon | UI component | photograph | composite screenshot | document\n"
        "- Identity: name the brand and the SPECIFIC version/generation if identifiable "
        "(e.g. 'legacy Google Pay mark with interlocking loops') — that generation label is often "
        "the decisive audit discriminator. Base every visual claim only on what is visible.\n"
        "- Visual signatures: exact shapes/outlines, typography (exact spelling, case, weight), "
        "sub-element layout, colours and gradients, design generation (legacy vs modern, flat vs 3D).\n"
        "- Person: describe facial bone structure ONLY (face shape, eye spacing, nose bridge, "
        "jawline, permanent features). Ignore clothing, hair colour, background.\n\n"
        "STEP 1 - DETECT SEARCH MODE\n"
        "Classify into exactly one mode:\n"
        "- LOGO_SIMILARITY: finding brand logos, icons, or visual identity marks similar to the reference\n"
        "- UI_COMPONENT: finding specific UI elements (buttons, modals, banners, cookie notices, payment flows)\n"
        "- PERSON_SEARCH: finding images where a specific person appears - even as a small profile thumbnail\n"
        "- EXACT_MATCH: finding near-identical or duplicate copies of the reference image (reverse image search)\n"
        "- GENERAL_ASSET: any other visual asset search\n\n"
        "STEP 2 - EXTRACT VISUAL CRITERIA\n"
        "Based STRICTLY on what is visible in the reference image and the user goal:\n"
        "- Every criterion must be verifiable by LOOKING at the two images alone - never metadata.\n"
        "- inclusion_criteria: Specific visual properties that MUST be present (shapes, exact text, layout, structure)\n"
        "- exclusion_criteria: What disqualifies a candidate (lookalikes, wrong brand variants, wrong generation, different subjects)\n"
        "- For PERSON_SEARCH: focus inclusion_criteria on facial bone structure, NOT clothing/background/color\n"
        "- For EXACT_MATCH: require near-pixel-level visual identity (same composition, same subject)\n"
        "- For LOGO_SIMILARITY / UI_COMPONENT: color is NOT a criterion unless the user goal explicitly requires it\n\n"
        "STEP 3 - GENERATE ADJUDICATION PROTOCOL\n"
        "Write adjudication_logic as 5 numbered mechanical steps modelling how a careful human "
        "reviewer inspects an image. The inference LLM always receives: IMAGE 0 (reference/target), "
        "IMAGE 1 (candidate), and this protocol. It MUST directly compare IMAGE 0 against IMAGE 1.\n"
        "Template - adapt every bracketed part to THIS audit goal and search mode:\n"
        "  STEP 1 (LOCATE): In IMAGE 0, identify the exact target element and its defining features: "
        "[shape / exact wordmark text / face bone structure / composition].\n"
        "  STEP 2 (SCAN): Sweep IMAGE 1 region by region (top-left to bottom-right), including inside "
        "screenshots, banners, app frames and thumbnails. Zoom into small and corner regions - the "
        "target may be tiny, cropped, low-contrast or partially occluded.\n"
        "  STEP 3 (COMPARE): Place the element located in IMAGE 1 beside IMAGE 0 and verify feature "
        "by feature: [exact structural checks - outline, typography spelling/weight, sub-element "
        "layout, generation old-vs-new, face jawline/eye-spacing/nose].\n"
        "  STEP 4 (EXCLUDE): Rule out lookalikes: [misspelled wordmarks, sibling brands, wrong "
        "generation, different person with similar features].\n"
        "  STEP 5 (VERDICT): PASS only if [exact condition confirmed by the IMAGE 0 vs IMAGE 1 "
        "comparison]. FAIL if the target is absent OR any exclusion triggers OR confidence is insufficient.\n"
        "Two different auditors following these steps must reach the same verdict.\n\n"
        "STEP 4 - GENERATE SEARCH SIGNALS\n"
        "vision_tag_filter (CRITICAL RULE):\n"
        f"- Select tags ONLY from this exact list of tags available in the database: [{tags_str}]\n"
        "- Do NOT invent tags. Select the 2-5 tags most specific to the audit target.\n"
        "- Prefer specific tags over generic ones (e.g. 'Trademark' over 'Text').\n"
        "- These tags scope the tag-search arm to only images Cloud Vision has labelled as relevant.\n\n"
        "search_keywords: distinctive single words only - they are OR-combined in search, so each "
        "must be discriminative alone. Matched against Gemini-generated image descriptions in the database.\n\n"
        "audit_instructions: Concise LLM guidance (120 words max). Do not repeat criteria.\n"
    )

    contents = [reference_image_part, prompt] if reference_image_part is not None else prompt
    response = client.models.generate_content(
        model=GEMINI_ORCHESTRATOR_MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=AuditContextModel,
            temperature=0.0
        )
    )
    return json.loads(response.text)


def build_fused_query_text(context_dict: dict, is_negative: bool = False) -> str:
    """
    Builds caption-style embedding text, mode-aware.
    Corpus vectors are IMAGE embeddings — query text must read like a description
    of the target image (a caption), never like instructions. Instruction prose
    ('Focus on...', 'Exclude images that...') lands far from every image vector.
    """
    search_mode = context_dict.get("search_mode", "GENERAL_ASSET")

    if is_negative:
        excl = ". ".join(context_dict.get("exclusion_criteria", []))
        return (excl or "Unrelated lookalike asset")[:1000]

    parts = []
    if context_dict.get("image_description"):
        parts.append(context_dict["image_description"][:600])

    if search_mode == "PERSON_SEARCH":
        parts.append(
            "A photo or screenshot containing this person's face at any size, "
            "including a small profile picture or thumbnail in a corner."
        )
    elif search_mode == "EXACT_MATCH":
        parts.append("An identical copy of this exact image.")
    else:
        parts.append(str(context_dict.get("audit_goal", "")))
        incl = "; ".join(context_dict.get("inclusion_criteria", []))
        if incl:
            parts.append(incl)

    return ". ".join(p for p in parts if p)[:1000]


async def embed_audit_context(context_dict: dict,
                               reference_image_path: Optional[str] = None,
                               face_crop_bytes: Optional[bytes] = None
                               ) -> Tuple[List[float], Optional[List[float]]]:
    """
    Generates POSITIVE embedding and optionally a NEGATIVE embedding.
    POSITIVE:
      - PERSON_SEARCH + face crop: embed the FACE CROP ALONE (vector anchored to the face)
      - reference image given:     embed the IMAGE ALONE (same space as corpus image vectors)
      - no image (text-to-image):  embed caption-style query text
    NEGATIVE: caption-style exclusion text; only for modes with weight > 0 - no wasted API call.
    The negative is folded into the query vector in Stage 2 (Rocchio), so the
    vector arm stays a single indexable ANN query.
    """
    search_mode = context_dict.get("search_mode", "GENERAL_ASSET")
    neg_weight = NEGATIVE_VECTOR_WEIGHT.get(search_mode, 0.0)
    needs_negative = neg_weight > 0.0

    neg_text = build_fused_query_text(context_dict, is_negative=True) if needs_negative else None

    pos_contents = []
    if search_mode == "PERSON_SEARCH" and face_crop_bytes:
        # Embed face crop: embedding is closer to other face images - higher recall for small faces
        pos_contents.append(types.Part.from_bytes(data=face_crop_bytes, mime_type="image/jpeg"))
    elif reference_image_path:
        mime = get_image_mime_type(reference_image_path)
        if reference_image_path.startswith("gs://"):
            pos_contents.append(types.Part.from_uri(file_uri=reference_image_path, mime_type=mime))
        else:
            with open(reference_image_path, "rb") as f:
                pos_contents.append(types.Part.from_bytes(data=f.read(), mime_type=mime))
    else:
        pos_contents.append(build_fused_query_text(context_dict, is_negative=False)[:1000])

    def _embed(contents):
        return client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=contents,
            config=types.EmbedContentConfig(output_dimensionality=768)
        ).embeddings[0].values

    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=2) as executor:
        pos_future = loop.run_in_executor(executor, _embed, pos_contents)
        if needs_negative:
            neg_future = loop.run_in_executor(executor, _embed, [neg_text[:1000]])
            pos_vec, neg_vec = await asyncio.gather(pos_future, neg_future)
        else:
            pos_vec = await pos_future
            neg_vec = None

    return pos_vec, neg_vec


In [ ]:
# 2. Parallel Hybrid Search + CPU Reranker + Drop-Off Detection
#
# LATENCY + RECALL FIXES (same structure, same names):
#   - Vector arm: the contrastive `ORDER BY (d_pos) - w*(d_neg)` is arithmetic on
#     two distances — NO vector index can ever serve it, so every query brute-force
#     scanned all 99L rows. The negative is now FOLDED INTO THE QUERY VECTOR
#     (Rocchio: q = normalize(pos - w*neg)): identical lookalike suppression,
#     single `ORDER BY embedding <=> $1 LIMIT N` served by the HNSW index.
#   - hnsw.ef_search raised (guarded): pgvector HNSW returns at most ef_search rows
#     per scan. At the default (40), `LIMIT 3000` silently returns ~40 rows.
#   - FTS arm: plainto_tsquery ANDs every keyword — 5-8 ANDed words means a
#     description must contain ALL of them (near-zero matches, recall killer).
#     Keywords are now OR-combined via to_tsquery.
#   - Tag/Face arms now ORDER BY tag-overlap count. They previously returned in
#     arbitrary order, so RRF was fusing a random permutation (rank noise).
#   - CPU reranker: multiplicative boosts (up to 48x) let generic-tag/keyword-spam
#     items bury true vector matches. Replaced with BOUNDED ADDITIVE nudges
#     (max ~2.6x): vector similarity stays the dominant signal; the LLM audit
#     is the precision gate.
#   - Python-level dedup (content_hash) unchanged — only unique images reach the
#     LLM (99L rows -> ~6L unique; inference cost stays on the unique set).
#
# DB indexes to create for further speedup (run once):
#   CREATE INDEX CONCURRENTLY ON visual_assets USING hnsw (embedding vector_cosine_ops);
#   CREATE INDEX CONCURRENTLY ON visual_assets USING GIN(vision_tags);
#   CREATE INDEX CONCURRENTLY ON visual_assets USING GIN(
#       to_tsvector('english', gemini_description));
#
from typing import List, Tuple, Optional
import math
import re
import numpy as np
import pandas as pd
from pgvector.asyncpg import register_vector
import asyncio

# Per-arm candidate caps — tune these if arms are still slow
VECTOR_ARM_LIMIT = 3000   # exact-scan cap; with HNSW active the scan returns up to ef_search rows
FTS_ARM_LIMIT    = 500    # FTS is slower; fewer results still covers relevant set
TAG_ARM_LIMIT    = 1000   # GIN-indexed if index exists
FACE_ARM_LIMIT   = 1000   # GIN-indexed if index exists


def compute_enhanced_local_reranker(candidates: list, audit_context: dict) -> list:
    """
    Zero-latency CPU multi-signal reranker. No LLM calls. Runs in < 5ms.

    All signals are BOUNDED ADDITIVE nudges on the RRF score (total multiplier
    <= ~2.6x). The reranker only decides WHO REACHES THE LLM, so recall and
    stability beat aggressiveness — the LLM audit is the precision gate.

    Signal 1: Vision tag match    - vision_tag_filter coverage on DB vision_tags   (+0.20 max)
    Signal 2: Keyword hits        - whole-word search_keywords in gemini_description (+0.10 max)
    Signal 3: Vector proximity    - exp(-3*distance), continuous                    (+0.30 max)
    Signal 4: Face-arm promotion  - PERSON_SEARCH face-tagged items                 (+0.40)
    Signal 5: Near-exact match    - EXACT_MATCH with vector_distance < 0.05         (+1.50)
    """
    search_mode     = audit_context.get("search_mode", "GENERAL_ASSET")
    tag_filter      = [t.lower().strip() for t in audit_context.get("vision_tag_filter", []) if t]
    search_keywords = [k.lower().strip() for k in audit_context.get("search_keywords", []) if k]

    for item in candidates:
        row        = item["data"]
        base_score = item.get("base_rrf_score", item["score"])

        desc_words = set(re.findall(r"[a-z0-9]+", str(row.get("gemini_description") or "").lower()))
        tags       = [str(t).lower() for t in (row.get("vision_tags") or [])]
        vec_dist   = item.get("vector_distance", 1.0)

        # Signal 1: fraction of LLM-chosen tags present on the asset (bounded 0..1)
        tag_hits  = sum(1 for ft in tag_filter if any(ft in t or t in ft for t in tags))
        tag_score = (tag_hits / len(tag_filter)) if tag_filter else 0.0

        # Signal 2: whole-word keyword hits (substring counting inflated 'art' via 'part')
        kw_hits  = sum(1 for kw in search_keywords if kw in desc_words)
        kw_score = (kw_hits / len(search_keywords)) if search_keywords else 0.0

        # Signal 3: smooth vector proximity (0..1)
        vec_prox = math.exp(-3.0 * vec_dist)

        bonus = 0.20 * tag_score + 0.10 * kw_score + 0.30 * vec_prox

        # Signal 4: Face-arm promotion (PERSON_SEARCH only)
        if search_mode == "PERSON_SEARCH" and item.get("promoted_by_face_arm"):
            bonus += 0.40

        # Signal 5: Near-exact match (EXACT_MATCH only)
        if search_mode == "EXACT_MATCH" and vec_dist < 0.05:
            bonus += 1.50

        score = base_score * (1.0 + bonus)
        item["score"]             = score
        item["base_rrf_score"]    = base_score
        item["rerank_multiplier"] = score / max(base_score, 1e-9)

    candidates.sort(key=lambda x: x["score"], reverse=True)
    return candidates


async def run_hybrid_search(scope_config: dict, audit_context: dict,
                             reference_image_path: Optional[str] = None,
                             face_crop_bytes: Optional[bytes] = None,
                             limit: int = 10000) -> List[dict]:
    """
    Parallel 4-Arm Hybrid Search with RRF fusion and zero-latency CPU reranking.

    Arms:
      1. Vector - bare HNSW `ORDER BY embedding <=> $1 LIMIT N` (negative folded
                  into the query vector, so the index always serves the query)
      2. FTS    - GIN-indexed tsvector query, keywords OR-combined
      3. Tags   - `&& $1::text[]` array overlap, ranked by overlap count
      4. Face   - PERSON_SEARCH only, tag overlap ranked by overlap count

    Python dedup (content_hash) runs during RRF merge — O(N) on small N,
    ensures only unique images go to LLM inference.
    """
    search_mode = audit_context.get("search_mode", "GENERAL_ASSET")
    weights     = RRF_WEIGHTS.get(search_mode, RRF_WEIGHTS["GENERAL_ASSET"])
    neg_weight  = NEGATIVE_VECTOR_WEIGHT.get(search_mode, 0.0)

    pos_vec, neg_vec = await embed_audit_context(
        audit_context, reference_image_path, face_crop_bytes
    )

    # Fold the negative into the query vector (Rocchio). Same lookalike
    # suppression as distance subtraction, but the SQL stays index-servable.
    if neg_vec is not None and neg_weight > 0.0:
        q = np.asarray(pos_vec, dtype=np.float32) - neg_weight * np.asarray(neg_vec, dtype=np.float32)
        query_vec = (q / (np.linalg.norm(q) + 1e-9)).tolist()
    else:
        query_vec = list(pos_vec)

    engine, _ = await get_alloydb_connection()

    search_keywords = audit_context.get("search_keywords", [])
    # OR-combined FTS query (plainto_tsquery ANDs all words -> near-zero recall)
    fts_terms       = [re.sub(r"[^A-Za-z0-9]", "", k) for k in search_keywords]
    fts_query_str   = " | ".join(t for t in fts_terms if t)
    # vision_tag_filter = LLM-chosen exact tag names from DB vocabulary
    # Using && (array overlap) — can use GIN index on vision_tags column
    tag_filter = audit_context.get("vision_tag_filter", [])

    # ── Arm 1: Vector Search ─────────────────────────────────────────────────
    # Bare HNSW query — no CTE, no subquery, no distance arithmetic in ORDER BY.
    # vector_distance is always the distance to the PURE POSITIVE vector, so the
    # safeguard thresholds / reranker / EXACT_MATCH logic keep clean semantics
    # even when the contrastive (folded) vector does the ordering.
    async def run_vector():
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            await register_vector(db)
            try:
                # HNSW returns at most ef_search rows per scan (default 40!)
                await db.execute("SET hnsw.ef_search = 1000")
            except Exception:
                pass  # parameter absent when this instance has no pgvector-HNSW
            if neg_vec and neg_weight > 0.0:
                # Contrastive mode: folded query vector orders; positive distance reported
                rows = await db.fetch(
                    f"SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description,"
                    f"       asset_filename, page_url, content_hash,"
                    f"       (embedding <=> $2::vector) AS vector_distance"
                    f" FROM {DB_SCHEMA}.visual_assets"
                    f" ORDER BY embedding <=> $1::vector ASC"
                    f" LIMIT $3",
                    query_vec, pos_vec, VECTOR_ARM_LIMIT
                )
            else:
                # Pure HNSW — fastest path, maximum recall
                rows = await db.fetch(
                    f"SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description,"
                    f"       asset_filename, page_url, content_hash,"
                    f"       (embedding <=> $1::vector) AS vector_distance"
                    f" FROM {DB_SCHEMA}.visual_assets"
                    f" ORDER BY embedding <=> $1::vector ASC"
                    f" LIMIT $2",
                    query_vec, VECTOR_ARM_LIMIT
                )
            return [dict(r) for r in rows]

    # ── Arm 2: Full-Text Search ───────────────────────────────────────────────
    # Needs GIN index: CREATE INDEX CONCURRENTLY ON visual_assets
    #   USING GIN(to_tsvector('english', gemini_description));
    # Keywords are OR-combined ('gpay | google | wallet') and ranked by ts_rank_cd,
    # so ONE strong keyword hit is enough to surface a candidate (recall), while
    # multi-keyword hits still rank higher (precision).
    async def run_fts():
        if not fts_query_str:
            return []
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                f"SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description,"
                f"       asset_filename, page_url, content_hash"
                f" FROM {DB_SCHEMA}.visual_assets"
                f" WHERE to_tsvector('english', gemini_description)"
                f"       @@ to_tsquery('english', $1)"
                f" ORDER BY ts_rank_cd("
                f"     to_tsvector('english', gemini_description),"
                f"     to_tsquery('english', $1)) DESC"
                f" LIMIT $2",
                fts_query_str, FTS_ARM_LIMIT
            )
            return [dict(r) for r in rows]

    # ── Arm 3: Vision Tag Filter ──────────────────────────────────────────────
    # vision_tag_filter = LLM-chosen exact tag names from top-300 DB frequency list.
    # Scopes this arm to only images Cloud Vision has labelled as relevant.
    # ORDER BY overlap count: RRF fuses RANKS, so an unranked arm injects noise.
    # Needs: CREATE INDEX CONCURRENTLY ON visual_assets USING GIN(vision_tags);
    async def run_tags():
        if not tag_filter:
            return []
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                f"SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description,"
                f"       asset_filename, page_url, content_hash"
                f" FROM {DB_SCHEMA}.visual_assets"
                f" WHERE vision_tags && $1::text[]"
                f" ORDER BY (SELECT COUNT(*) FROM unnest(vision_tags) AS vt"
                f"           WHERE vt = ANY($1::text[])) DESC"
                f" LIMIT $2",
                tag_filter, TAG_ARM_LIMIT
            )
            return [dict(r) for r in rows]

    # ── Arm 4: Face Tag Arm — PERSON_SEARCH only ─────────────────────────────
    # Broad net for human presence, ranked by face-tag overlap count.
    # Needs: CREATE INDEX CONCURRENTLY ON visual_assets USING GIN(vision_tags);
    async def run_face_arm():
        if search_mode != "PERSON_SEARCH":
            return []
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                f"SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description,"
                f"       asset_filename, page_url, content_hash"
                f" FROM {DB_SCHEMA}.visual_assets"
                f" WHERE vision_tags && $1::text[]"
                f" ORDER BY (SELECT COUNT(*) FROM unnest(vision_tags) AS vt"
                f"           WHERE vt = ANY($1::text[])) DESC"
                f" LIMIT $2",
                PERSON_SEARCH_FACE_TAGS, FACE_ARM_LIMIT
            )
            return [dict(r) for r in rows]

    # Run all arms concurrently
    t_arms_start = asyncio.get_event_loop().time()
    vector_results, fts_results, tag_results, face_results = await asyncio.gather(
        run_vector(), run_fts(), run_tags(), run_face_arm()
    )
    t_arms = asyncio.get_event_loop().time() - t_arms_start
    print(f"   Arms ({t_arms:.2f}s): Vector={len(vector_results)} | FTS={len(fts_results)}"
          f" | Tags={len(tag_results)} | Face={len(face_results)}")

    # Track promoted IDs for drop-off rescue
    promoted_ids = set()
    face_arm_ids = set()
    for r in fts_results:   promoted_ids.add(str(r["asset_id"]))
    for r in tag_results:   promoted_ids.add(str(r["asset_id"]))
    for r in face_results:
        face_arm_ids.add(str(r["asset_id"]))
        promoted_ids.add(str(r["asset_id"]))

    # ── RRF Fusion (k=60) ────────────────────────────────────────────────────
    k = 60
    results_map = {}
    vector_distance_map = {str(r["asset_id"]): r.get("vector_distance", 1.0) for r in vector_results}

    def upsert_ranks(results_list, weight: float):
        for rank, row in enumerate(results_list):
            img_id = str(row["asset_id"])
            if img_id not in results_map:
                results_map[img_id] = {
                    "data": row, "score": 0.0,
                    "vector_distance": vector_distance_map.get(img_id, 1.0)
                }
            results_map[img_id]["score"] += weight / (k + rank + 1)

    upsert_ranks(vector_results, weights["vector"])
    if fts_results:  upsert_ranks(fts_results,  weights["fts"])
    if tag_results:  upsert_ranks(tag_results,  weights["tags"])
    if face_results: upsert_ranks(face_results, weights["face"])

    fused = list(results_map.values())

    # Apply zero-latency CPU reranker
    reranked = compute_enhanced_local_reranker(
        [{**item, "base_rrf_score": item["score"],
          "promoted_by_face_arm": str(item["data"]["asset_id"]) in face_arm_ids}
         for item in fused],
        audit_context
    )

    # Python-level dedup by content_hash — O(N) on merged set, fast.
    # Ensures only unique images reach LLM inference (saves inference cost).
    seen = set()
    deduplicated = []
    for item in reranked:
        row        = item["data"]
        img_id     = str(row["asset_id"])
        identifier = row.get("content_hash") or row.get("gcs_raw_path")
        if identifier not in seen:
            seen.add(identifier)
            deduplicated.append({
                **row,
                "relevance_score":            item["score"],
                "base_rrf_score":             item.get("base_rrf_score", 0),
                "rerank_multiplier":          item.get("rerank_multiplier", 1),
                "vector_distance":            item.get("vector_distance", 1.0),
                "promoted_by_keyword_or_tag": img_id in promoted_ids,
                "promoted_by_face_arm":       img_id in face_arm_ids,
            })

    print(f"   After dedup: {len(deduplicated)} unique candidates")
    return deduplicated[:limit]


def detect_dropoff_flawless(df: pd.DataFrame,
                              search_mode: str = "GENERAL_ASSET",
                              sensitivity: Optional[float] = None
                              ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Segments candidates into High / Edge / Low tiers using Kneedle curvature
    + rolling volatility. All thresholds and sensitivities are mode-aware.
    Three rescue passes protect recall after segmentation. Rescue passes 2 and 3
    are BOUNDED (top MAX_CANDIDATES_QUICK_MODE by score): an unbounded rescue
    could dump an entire 1000-row arm into Edge and blow up LLM cost/latency.
    """
    if df is None or df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    if sensitivity is None:
        sensitivity = KNEEDLE_SENSITIVITY.get(search_mode, 1.0)
    safeguard_dist = VECTOR_SAFEGUARD_THRESHOLD.get(search_mode, 0.35)

    df_sorted = df.sort_values("relevance_score", ascending=False).reset_index(drop=True)
    y = df_sorted["relevance_score"].values
    n = len(y)

    y_min, y_max = y.min(), y.max()
    if y_max == y_min:
        s1, s2 = int(n * 0.3), int(n * 0.6)
        return df_sorted.iloc[:s1], df_sorted.iloc[s1:s2], df_sorted.iloc[s2:]

    y_norm = (y - y_min) / (y_max - y_min + 1e-9)
    x_norm = np.arange(n) / max(n - 1, 1)
    coords = np.column_stack((x_norm, y_norm))

    # Kneedle: max perpendicular distance from diagonal
    line_vec = coords[-1] - coords[0]
    line_len = np.sqrt(np.sum(line_vec ** 2))
    if line_len < 1e-9:
        idx1 = n // 3
    else:
        lv = line_vec / line_len
        dist_to_line = np.sqrt(np.sum(
            (coords - np.outer(np.dot(coords - coords[0], lv), lv)) ** 2, axis=1
        ))
        idx1 = int(np.argmax(dist_to_line))

    floor_size = max(15, int(n * 0.05))
    idx1 = max(idx1, floor_size)

    # Rolling volatility: find the quiet tail (Edge -> Low boundary)
    window = max(3, int(n * 0.05))
    rolling_std = pd.Series(y_norm).rolling(window=window, center=True).std().fillna(0).values
    noise_threshold = np.mean(rolling_std) * (0.6 / sensitivity)
    idx2 = n - 1
    for i in range(idx1 + 2, len(rolling_std)):
        if rolling_std[i] < noise_threshold:
            idx2 = i
            break

    min_edge_width = max(10, int((n - idx1) * 0.25))
    if (idx2 - idx1) < min_edge_width:
        idx2 = min(n - 1, idx1 + min_edge_width)

    high_df = df_sorted.iloc[:idx1 + 1].copy()
    edge_df = df_sorted.iloc[idx1 + 1:idx2 + 1].copy()
    low_df  = df_sorted.iloc[idx2 + 1:].copy()

    # Rescue 1: Vector distance safeguard (mode-aware, naturally bounded by threshold)
    if "vector_distance" in low_df.columns:
        rescued = low_df[low_df["vector_distance"] < safeguard_dist].copy()
        if not rescued.empty:
            edge_df = pd.concat([edge_df, rescued], ignore_index=True)
            low_df  = low_df[low_df["vector_distance"] >= safeguard_dist].copy()
            print(f"   Shield [{search_mode}]: rescued {len(rescued)} by vec_dist < {safeguard_dist}")

    # Rescue 2: Face-arm items (PERSON_SEARCH) — best MAX_CANDIDATES_QUICK_MODE by score
    if search_mode == "PERSON_SEARCH" and "promoted_by_face_arm" in low_df.columns:
        rescued = (low_df[low_df["promoted_by_face_arm"] == True]
                   .sort_values("relevance_score", ascending=False)
                   .head(MAX_CANDIDATES_QUICK_MODE))
        if not rescued.empty:
            edge_df = pd.concat([edge_df, rescued.copy()], ignore_index=True)
            low_df  = low_df.drop(rescued.index)
            print(f"   Shield [Face]: rescued {len(rescued)} face-tagged items (bounded)")

    # Rescue 3: Keyword/tag promoted items — best MAX_CANDIDATES_QUICK_MODE by score
    if "promoted_by_keyword_or_tag" in low_df.columns:
        rescued = (low_df[low_df["promoted_by_keyword_or_tag"] == True]
                   .sort_values("relevance_score", ascending=False)
                   .head(MAX_CANDIDATES_QUICK_MODE))
        if not rescued.empty:
            edge_df = pd.concat([edge_df, rescued.copy()], ignore_index=True)
            low_df  = low_df.drop(rescued.index)
            print(f"   Shield [KW/Tag]: rescued {len(rescued)} keyword/tag promoted items (bounded)")

    print(f"   Tiers: High={len(high_df)} | Edge={len(edge_df)} | Low={len(low_df)}")
    return high_df, edge_df, low_df


## Retrieval, Reranking, and Drop-off Segmentation Engine

This cell defines the core search engine:

- `run_hybrid_search`: Combines Vector, Full-Text, and Tag search arms into a fused RRF list, applying zero-latency keyword boosts.
- `detect_dropoff_flawless`: Curvature-based Kneedle algorithm that truncates irrelevant tail results.

In [ ]:
# 3. Hydration & Parallel LLM Audit Inference
#
# Each inference call sends the LLM an explicitly LABELLED sequence:
#   "IMAGE 0 = REFERENCE"  -> IMAGE 0 Part (target template or person)
#   "IMAGE 1 = CANDIDATE"  -> IMAGE 1 Part (asset under evaluation)
#   TEXT                   -> persona + audit context block (criteria +
#                             adjudication protocol) + mode-aware inspection method
#
# PROMPT FIXES (same structure, same names):
#   - Criteria + adjudication were previously injected TWICE per prompt (context
#     block AND again inside each mode protocol) — duplication dilutes attention.
#     Single source now: the context block carries the contract; mode protocols
#     carry only the human-auditor inspection METHOD (scan pattern, zoom,
#     tiny/embedded targets, feature-by-feature IMAGE 0 vs IMAGE 1 comparison).
#   - Images are labelled inline so the model can never confuse reference/candidate.
#   - Transient API errors (429/5xx) retry with backoff instead of silently
#     becoming FAIL verdicts (false negatives that cost recall).
import asyncio
from concurrent.futures import ThreadPoolExecutor
import json
import pandas as pd
from typing import List, Tuple, Optional
from google.genai import types
from google.cloud import storage
from pydantic import create_model, Field
import time
from PIL import Image
import io


def process_transparency(image_bytes: bytes,
                          default_bg: Tuple[int, int, int] = (30, 30, 30)) -> bytes:
    """Composites transparent images onto a dark background so light elements stay visible."""
    try:
        img = Image.open(io.BytesIO(image_bytes))
        if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
            img = img.convert('RGBA')
            bg = Image.new("RGBA", img.size, default_bg + (255,))
            final_img = Image.alpha_composite(bg, img).convert("RGB")
            out = io.BytesIO()
            final_img.save(out, format="PNG")
            return out.getvalue()
    except Exception as e:
        print(f"Warning: transparency processing failed: {e}")
    return image_bytes


def build_audit_context_block(audit_config: dict) -> str:
    """
    Builds the structured AUDIT CONTEXT text block sent to the LLM alongside both images.
    This is the SINGLE SOURCE for criteria and the adjudication protocol —
    the mode protocols in build_audit_prompt never repeat them.
    """
    search_mode   = audit_config.get("search_mode", "GENERAL_ASSET")
    audit_goal    = audit_config.get("audit_goal", "")
    image_desc    = (audit_config.get("image_description") or "")[:300]
    inclusion_str = "\n".join(f"  - {c}" for c in audit_config.get("inclusion_criteria", [])) or "  - None"
    exclusion_str = "\n".join(f"  - {c}" for c in audit_config.get("exclusion_criteria", [])) or "  - None"
    adjudication  = audit_config.get("adjudication_logic", "")
    return (
        f"=== AUDIT CONTEXT (read before examining images) ===\n"
        f"Search Mode   : {search_mode}\n"
        f"Audit Goal    : {audit_goal}\n"
        f"Reference Desc: {image_desc if image_desc else 'See IMAGE 0'}\n\n"
        f"Inclusion Criteria (asset MUST meet ALL of these):\n{inclusion_str}\n\n"
        f"Exclusion Criteria (asset FAILS if ANY of these trigger):\n{exclusion_str}\n\n"
        f"Adjudication Protocol (execute these steps IN ORDER, recording findings per step):\n"
        f"{adjudication}\n"
        f"{'='*50}\n"
    )


def build_image_layout_block(has_reference: bool) -> str:
    """Explains the image layout to the LLM so it knows which image is which."""
    if has_reference:
        return (
            "IMAGE LAYOUT (each image is preceded by a text label naming it):\n"
            "  IMAGE 0 = Reference image uploaded by the user (target template or person).\n"
            "  IMAGE 1 = Candidate image under audit.\n\n"
            "ISOLATION RULE: IMAGE 0 may contain device frames, hands, backgrounds, or "
            "surrounding UI noise. Focus ONLY on the target element (logo, face, component) "
            "inside IMAGE 0 when comparing against IMAGE 1.\n\n"
        )
    return (
        "IMAGE LAYOUT:\n"
        "  IMAGE 0 = Candidate image under audit (no reference image provided).\n\n"
    )


def build_audit_prompt(audit_config: dict, has_reference: bool) -> str:
    """
    Builds a mode-aware structured audit prompt.
    The prompt is TEXT only - the images are passed separately as labelled Part objects.
    Structure: [persona] + [audit context block] + [image layout] + [inspection method].
    Criteria and adjudication live ONLY in the context block — no duplication.
    """
    search_mode  = audit_config.get("search_mode", "GENERAL_ASSET")
    instructions = audit_config.get("audit_instructions", "")

    persona = (
        "You are a meticulous visual asset auditor. You inspect an image the way a trained "
        "human reviewer does: you scan the whole canvas region by region, zoom into small "
        "and corner areas, and never assume the target is centered, isolated, or full-size. "
        "You compare the candidate against the reference feature by feature before deciding, "
        "and you follow the Adjudication Protocol steps literally.\n\n"
    )

    context_block = build_audit_context_block(audit_config)
    layout_block  = build_image_layout_block(has_reference)

    if search_mode == "PERSON_SEARCH":
        protocol = (
            "HOW TO INSPECT (face search) - apply while executing the Adjudication Protocol:\n"
            "1. Sweep IMAGE 1 systematically: top-left -> top-right -> center -> bottom-left -> "
            "bottom-right. Inventory EVERY human face at ALL scales: full portraits, tiny profile "
            "thumbnails (even ~30px), faces in crowds, faces inside app screens or screenshots, "
            "faces partially occluded or cropped at edges. List each with position and approximate "
            "size. Zero faces found -> matches_criteria = False.\n"
            "2. For each face, compare bone structure against IMAGE 0: face shape and jawline; "
            "eye spacing, size and shape; nose bridge and tip; forehead height and hairline; "
            "permanent distinctive features (brow ridge, ears, cleft chin). "
            "IGNORE clothing, hair colour/style, skin-tone rendering, background, lighting, angle.\n"
            "3. A small or low-resolution face is NOT a FAIL by itself: judge on the features "
            "that ARE visible and reflect the reduced certainty in match_confidence.\n"
            "4. Confidence bands: >=80 clear match (multiple structural features confirmed); "
            "50-79 probable match (occlusion, extreme angle, or very small size); "
            "<50 uncertain or clearly different.\n"
            "5. matches_criteria = True if ANY face matches the reference person per the "
            "Adjudication Protocol; False if no face matches or all are too degraded.\n\n"
        )
    elif search_mode == "EXACT_MATCH":
        protocol = (
            "HOW TO INSPECT (reverse image match) - apply while executing the Adjudication Protocol:\n"
            "1. Compare the overall composition of IMAGE 1 against IMAGE 0: main subject, "
            "layout/framing, dominant visual elements.\n"
            "2. Decide near-identical vs merely similar. Acceptable differences: resolution change, "
            "compression artifacts, slight crop/padding, minor colour shift, format conversion. "
            "NOT acceptable: different subject, composition, context, product or scene - even if "
            "from the same brand or series.\n"
            "3. matches_criteria = True only for a near-identical copy of the same underlying "
            "image (possibly re-encoded).\n\n"
        )
    elif search_mode in ("LOGO_SIMILARITY", "UI_COMPONENT"):
        protocol = (
            "HOW TO INSPECT (logo / UI component) - apply while executing the Adjudication Protocol:\n"
            "1. Inventory IMAGE 1 fully: every logo, icon, wordmark, button, banner and layout "
            "region, with positions - including elements embedded inside composite screenshots, "
            "hero banners, app frames or mockups. The target may appear at ANY scale, cropped, or "
            "low-contrast; absence of an ISOLATED version is NOT a FAIL - locate before judging.\n"
            "2. Compare the located element side-by-side with the target element in IMAGE 0, "
            "feature by feature: shape, outline, proportions; typography with EXACT spelling and "
            "weight ('Pay' != 'Play', 'Ads' != 'AdWords'); arrangement of sub-elements; design "
            "generation (legacy vs modern, flat vs 3D, filled vs outlined).\n"
            "3. COLOR RULE: colour differences alone do NOT disqualify a match unless the "
            "inclusion criteria explicitly require a specific colour.\n"
            "4. Explicitly test every exclusion criterion: lookalike misspellings, sibling or "
            "related-but-different brands, deprecated or wrong-generation variants.\n"
            "5. matches_criteria = True only if the target is found AND the IMAGE 0 vs IMAGE 1 "
            "comparison passes every inclusion criterion AND no exclusion triggers. "
            "If borderline, lower match_confidence - the guardrail demotes low-confidence passes.\n\n"
        )
    else:  # GENERAL_ASSET
        protocol = (
            "HOW TO INSPECT (general asset) - apply while executing the Adjudication Protocol:\n"
            "1. Describe all visual elements of IMAGE 1 systematically, region by region, "
            "including small and embedded elements.\n"
            "2. Check relevance to the audit goal, then test every inclusion and exclusion "
            "criterion explicitly" + (" against IMAGE 0" if has_reference else "") + ".\n"
            "3. matches_criteria = True only if every inclusion criterion passes and no "
            "exclusion triggers.\n\n"
        )

    general = f"[General Instructions]:\n{instructions}\n" if instructions else ""
    return persona + context_block + layout_block + protocol + general


def enforce_zero_false_positives_rules(df: pd.DataFrame,
                                        search_mode: str = "GENERAL_ASSET") -> pd.DataFrame:
    """
    Mode-aware confidence guardrail. Demotes PASS verdicts below mode threshold to FAIL.
    Thresholds calibrated per mode to balance precision/recall simultaneously.
    """
    if df.empty:
        return df
    threshold = CONFIDENCE_THRESHOLD.get(search_mode, 70)

    def guardrail_check(row):
        matches = bool(row.get("matches_criteria", False))
        conf    = int(row.get("match_confidence", 100))
        if matches and conf < threshold:
            row["matches_criteria"] = False
            row["match_rationale"] = (
                f"[GUARDRAIL: conf {conf}% < {threshold}% threshold for {search_mode}] "
                + str(row.get("match_rationale", ""))
            )
        return row

    return df.apply(guardrail_check, axis=1)


async def run_llm_audit_single(asset_data: dict, audit_config: dict,
                                _executor=None,
                                reference_image_part: Optional[types.Part] = None) -> dict:
    """
    Evaluates a single image asset.
    Sends the LLM a labelled sequence:
      "IMAGE 0 = REFERENCE" -> IMAGE 0 Part (if available)
      "IMAGE 1 = CANDIDATE" -> IMAGE 1 Part
      Text prompt           -> persona + audit context + inspection method
    Transient API errors (429/5xx) retry with backoff before counting as failure.
    """
    search_mode       = audit_config.get("search_mode", "GENERAL_ASSET")
    extraction_schema = audit_config.get("extraction_schema", {})
    has_reference     = reference_image_part is not None

    fields = {
        "visual_analysis_step_by_step": (
            str,
            Field(description=(
                "MANDATORY CHAIN OF THOUGHT: Execute EVERY numbered step of the Adjudication "
                "Protocol IN ORDER and write your findings per step (STEP 1: ... STEP 2: ...) "
                "before reaching a verdict. For located elements, state WHERE in IMAGE 1 they "
                "appear and WHAT the IMAGE 0 vs IMAGE 1 feature comparison showed. "
                "Do NOT skip steps or jump directly to a verdict."
            ))
        ),
        "matches_criteria": (
            bool,
            Field(description=(
                "Final verdict. True ONLY if the asset satisfies ALL inclusion criteria "
                "and triggers NO exclusion criteria with sufficient confidence for this search mode."
            ))
        ),
        "match_confidence": (
            int,
            Field(description=(
                "Confidence percentage 0-100. Reduce below threshold if there is ANY "
                "visual ambiguity, occlusion, small size, or uncertainty. "
                "When in doubt, score lower - the guardrail handles threshold enforcement."
            ))
        ),
        "match_rationale": (
            str,
            Field(description=(
                "One concise paragraph citing specific visual evidence from your step-by-step analysis "
                "explaining exactly why matches_criteria is True or False."
            ))
        ),
    }

    for field_name, field_info in extraction_schema.items():
        if field_name in fields:
            continue
        t, desc = str, f"Extracted value for {field_name}"
        if isinstance(field_info, dict):
            ftype = field_info.get("field_type", "string").lower()
            desc  = field_info.get("description", desc)
        else:
            ftype = str(field_info).lower()
        if ftype == "boolean":             t = bool
        elif ftype in ("integer", "int"):  t = int
        elif ftype in ("number", "float"): t = float
        fields[field_name] = (t, Field(description=desc))

    DynamicAuditModel = create_model("DynamicAuditModel", **fields)

    # Build text prompt (persona + context block + layout block + inspection method)
    audit_prompt = build_audit_prompt(audit_config, has_reference)
    gcs_path = asset_data["gcs_raw_path"]

    def _download_and_preprocess():
        if gcs_path.startswith("gs://"):
            parts = gcs_path.replace("gs://", "").split("/", 1)
            c = storage.Client(project=PROJECT_ID)
            img_bytes = c.bucket(parts[0]).blob(parts[1]).download_as_bytes()
        else:
            with open(gcs_path, "rb") as f:
                img_bytes = f.read()
        return process_transparency(img_bytes)

    loop = asyncio.get_running_loop()
    try:
        processed_bytes = await loop.run_in_executor(_executor, _download_and_preprocess)
        candidate_part  = types.Part.from_bytes(data=processed_bytes, mime_type="image/png")

        # Labelled contents: the model can never confuse reference and candidate
        contents = []
        if reference_image_part:
            contents.append("IMAGE 0 = REFERENCE (the target):")
            contents.append(reference_image_part)                 # IMAGE 0
            contents.append("IMAGE 1 = CANDIDATE (under audit):")
        else:
            contents.append("IMAGE 0 = CANDIDATE (under audit):")
        contents.append(candidate_part)                           # IMAGE 1 (or 0 if no ref)
        contents.append(audit_prompt)                             # Audit context + protocol

        def _call_gemini():
            # Retry transient errors: one failed call otherwise becomes a silent
            # FAIL verdict — a false negative that costs recall.
            last_err = None
            for attempt in range(3):
                try:
                    return client.models.generate_content(
                        model=GEMINI_INFERENCE_MODEL,
                        contents=contents,
                        config=types.GenerateContentConfig(
                            response_mime_type="application/json",
                            response_schema=DynamicAuditModel,
                            temperature=0.0
                        )
                    )
                except Exception as e:
                    last_err = e
                    transient = any(t in str(e) for t in
                                    ("429", "500", "503", "RESOURCE_EXHAUSTED",
                                     "UNAVAILABLE", "DEADLINE", "INTERNAL"))
                    if transient and attempt < 2:
                        time.sleep(1.5 * (attempt + 1))
                        continue
                    raise
            raise last_err

        response       = await loop.run_in_executor(_executor, _call_gemini)
        extracted_data = json.loads(response.text)
    except Exception as e:
        extracted_data = {
            "matches_criteria":             False,
            "match_confidence":             0,
            "match_rationale":              f"Audit failed: {e}",
            "visual_analysis_step_by_step": f"ERROR: {e}",
            "error":                        str(e),
        }

    return {**asset_data, **extracted_data}


async def run_llm_inference_on_dropoff_results(df_high: pd.DataFrame,
                                                df_edge: pd.DataFrame,
                                                audit_config: dict,
                                                max_workers: int = MAX_INFERENCE_WORKERS,
                                                reference_image_path: Optional[str] = None
                                                ) -> pd.DataFrame:
    """
    Parallel LLM visual inference on High + Edge candidate tiers.
    Reference image loaded once and reused across all workers.
    Only unique images reach inference (dedup already ran in Stage 2).
    """
    candidates_df = pd.concat([df_high, df_edge], ignore_index=True)
    if candidates_df.empty:
        return pd.DataFrame()

    search_mode = audit_config.get("search_mode", "GENERAL_ASSET")
    candidates  = candidates_df.to_dict(orient="records")
    print(f"   LLM audit: {len(candidates)} unique candidates | Mode: {search_mode} | Workers: {max_workers}")

    def _get_reference_part():
        if not reference_image_path:
            return None
        if reference_image_path.startswith("gs://"):
            parts = reference_image_path.replace("gs://", "").split("/", 1)
            c = storage.Client(project=PROJECT_ID)
            ref_bytes = c.bucket(parts[0]).blob(parts[1]).download_as_bytes()
        else:
            with open(reference_image_path, "rb") as f:
                ref_bytes = f.read()
        return types.Part.from_bytes(data=process_transparency(ref_bytes), mime_type="image/png")

    t0   = time.time()
    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        reference_image_part = await loop.run_in_executor(executor, _get_reference_part)
        tasks   = [run_llm_audit_single(a, audit_config, executor, reference_image_part)
                   for a in candidates]
        results = await asyncio.gather(*tasks)

    dur        = time.time() - t0
    throughput = len(candidates) / max(dur, 0.01)
    print(f"   Done: {len(candidates)} assets | {dur:.1f}s | {throughput:.1f} img/s")

    results_df = pd.DataFrame(results)
    results_df = enforce_zero_false_positives_rules(results_df, search_mode)
    results_df = results_df.sort_values("relevance_score", ascending=False).reset_index(drop=True)

    print(f"\n=== AUDIT SCOREBOARD (Mode: {search_mode}) ===")
    for i, r in results_df.iterrows():
        status = "PASS" if r.get("matches_criteria") else "FAIL"
        fname  = r.get("asset_filename") or r.get("gcs_raw_path", "").split("/")[-1]
        print(f"  [{i+1}] {status} | {fname[:45]} | Score:{r.get('relevance_score',0):.4f} | Conf:{r.get('match_confidence',0)}%")

    return results_df


In [ ]:
# 4. Calibration Summary & Audit Results Saving E2E Loop
async def save_audit_results_to_db(session_id: str, results_df: pd.DataFrame):
    """Saves the final audited results into AlloyDB (Bypassed by default in the interactive runner)."""
    if results_df.empty:
        return

    engine, _ = await get_alloydb_connection()
    async with engine.connect() as conn:
        raw_conn = await conn.get_raw_connection()
        db = raw_conn.driver_connection
        records = results_df.to_dict("records")
        for r in records:
            verdict = "PASS" if r.get("matches_criteria") is True else "FAIL"

            # Extract criteria_checks from JSON response or construct it
            criteria_checks = {k: v for k, v in r.items() if k not in ["asset_id", "gcs_raw_path", "matches_criteria", "error", "asset_filename", "page_url"]}

            await db.execute(
                f"""
                INSERT INTO {DB_SCHEMA}.audit_results (
                    session_id, asset_id, overall_verdict, adjudication_result,
                    criteria_checks, rationale, confidence_band
                ) VALUES ($1, $2, $3, $4, $5, $6, $7)
                """,
                session_id,
                r.get("asset_id"),
                verdict,
                r.get("matches_criteria", False),
                json.dumps(criteria_checks),
                r.get("match_rationale", "Completed"),
                "high" if r.get("match_confidence", 0) > 95 else ("borderline" if r.get("match_confidence", 0) >= 70 else "below_threshold")
            )
    print("Audit results saved to database.")

async def generate_ai_audit_summary(results_df: pd.DataFrame, audit_config: dict) -> str:
    """Generates an AI-powered executive summary of the visual asset audit results."""
    import json
    import numpy as np
    if results_df is None or results_df.empty:
        return "No audit results available to generate a summary."

    # Select columns to pass to the LLM, avoiding internal or verbose vector columns
    exclude_cols = {"num_chunks", "max_relevance_score", "gcs_raw_path", "gcs_processed_path", "embedding", "embedding_at"}
    cols_to_include = [col for col in results_df.columns if col not in exclude_cols]

    # Convert to records safely, handling NumPy arrays, lists, and floats without boolean truth value ambiguity
    clean_df = results_df[cols_to_include].copy()
    for col in clean_df.columns:
        def safe_clean(val):
            if val is None:
                return None
            if isinstance(val, (np.ndarray, pd.Series)):
                return val.tolist() if val.size > 0 else None
            if isinstance(val, float) and pd.isna(val):
                return None
            return val
        clean_df[col] = clean_df[col].apply(safe_clean)

    records = clean_df.to_dict(orient="records")
    formatted_results = json.dumps(records, indent=2)

    audit_instructions = audit_config.get("audit_instructions", "No specific audit context provided.")

    # Compute basic stats to seed in the prompt
    total_audited = len(results_df)
    matches_col = "matches_criteria" if "matches_criteria" in results_df.columns else None
    if matches_col:
        # Convert to boolean safely, handling string representation if any
        matches_true = results_df[matches_col].apply(lambda x: str(x).lower() in ("true", "1", "yes")).sum()
    else:
        matches_true = "N/A"

    summary_prompt = f"""
You are a Lead Visual Asset Auditor.
Your task is to write a visually engaging, highly structured, and extremely concise summary of a visual asset audit Test Bench calibration run.

STRICT RULES FOR FORMATTING & SECTIONS:
1. You MUST only include the following exact three sections in the output:
   - Objective & Scope (Preview Subset) (within the top [!NOTE] block)
   - Calibration Statistics (as a numbered list)
   - Configuration Calibration Insights (as a single, brief narrative paragraph of 3-4 sentences detailing the visual rules performance)
2. DO NOT include any other sections.
3. DO NOT pass any definitive verdicts of success.

AUDIT CONTEXT (Visual Evaluation Criteria):
{audit_instructions}

TEST BENCH STATS:
- Total images audited: {total_audited}
- Total images matching criteria (True): {matches_true}

TEST BENCH FINDINGS (JSON):
{formatted_results}
"""

    response = client.models.generate_content(
        model=GEMINI_ORCHESTRATOR_MODEL,
        contents=summary_prompt,
        config=types.GenerateContentConfig(temperature=0.0)
    )
    return response.text


## Telemetry Summaries & Calibration Database Saving

This cell defines functions to generate final executive AI summaries (`generate_ai_audit_summary`) and save audit session details into the run calibration tables in AlloyDB for telemetry history.

In [ ]:
# 5. E2E Execution Helpers
#
# Key changes from original:
#   - 70k tag fix: queries top-{MAX_TAGS_TO_LLM} most frequent tags by COUNT(*) not SELECT DISTINCT
#   - Stage 1 is now ONE Gemini Pro call: forensic analysis + config generation merged
#     (halves Stage-1 latency and cost; image_description carries the forensic report)
#   - Cloud Vision face detection runs once at Stage 1 (zero per-candidate overhead)
#   - Smart Scan capped at MAX_CANDIDATES_SMART_SCAN (High tier first, Edge fills the
#     remainder) so LLM cost and latency stay bounded and predictable
#   - All pipeline stages are mode-aware; only unique images reach inference
import base64, os, time, json
import pandas as pd
from google.cloud import storage


def get_gcs_image_base64(gcs_path: str) -> str:
    """Downloads image from GCS or local, returns base64 data URI for HTML display."""
    try:
        mime = "image/png"
        low  = gcs_path.lower()
        if low.endswith((".jpg", ".jpeg")): mime = "image/jpeg"
        elif low.endswith(".webp"):          mime = "image/webp"
        elif low.endswith(".gif"):           mime = "image/gif"
        if gcs_path.startswith("gs://"):
            parts = gcs_path.replace("gs://", "").split("/", 1)
            img_bytes = storage.Client().bucket(parts[0]).blob(parts[1]).download_as_bytes()
        elif os.path.exists(gcs_path):
            with open(gcs_path, "rb") as f:
                img_bytes = f.read()
        else:
            return ""
        return f"data:{mime};base64,{base64.b64encode(img_bytes).decode()}"
    except Exception:
        return ""


def extract_face_crop(reference_image_path: str) -> Optional[bytes]:
    """
    Runs Cloud Vision Face Detection on the reference image and returns cropped face bytes.
    Called ONCE at Stage 1 - zero per-candidate overhead in the pipeline.
    The face crop is used for embedding (not full image) so that the vector embedding
    is anchored to the face itself, dramatically improving recall for small/corner appearances.
    Returns None if no face detected or on error.
    """
    try:
        from google.cloud import vision
        vc = vision.ImageAnnotatorClient()
        if reference_image_path.startswith("gs://"):
            image = vision.Image(source=vision.ImageSource(gcs_image_uri=reference_image_path))
        else:
            with open(reference_image_path, "rb") as f:
                image = vision.Image(content=f.read())

        resp = vc.face_detection(image=image)
        if not resp.face_annotations:
            print("   No face detected in reference image - using full image for embedding.")
            return None

        face  = resp.face_annotations[0]  # Highest-confidence face
        verts = face.bounding_poly.vertices
        x_min = max(min(v.x for v in verts) - 20, 0)
        y_min = max(min(v.y for v in verts) - 20, 0)
        x_max = max(v.x for v in verts) + 20
        y_max = max(v.y for v in verts) + 20

        if reference_image_path.startswith("gs://"):
            parts = reference_image_path.replace("gs://", "").split("/", 1)
            img_bytes = storage.Client(project=PROJECT_ID).bucket(parts[0]).blob(parts[1]).download_as_bytes()
        else:
            with open(reference_image_path, "rb") as f:
                img_bytes = f.read()

        from PIL import Image as PILImage
        img  = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")
        crop = img.crop((x_min, y_min, x_max, y_max))
        out  = io.BytesIO()
        crop.save(out, format="JPEG", quality=95)
        print(f"   Face crop: {x_max-x_min}x{y_max-y_min}px - used for person embedding.")
        return out.getvalue()
    except Exception as e:
        print(f"   Face detection failed ({e}). Using full image for embedding.")
        return None


async def generate_audit_config_only(user_goal: str,
                                      reference_image_path: Optional[str] = None) -> dict:
    """
    Stage 1: ONE Gemini Pro call - forensic reference analysis + structured Audit
    Configuration generated together (was two sequential Pro calls; merging halves
    Stage-1 latency and cost). image_description in the config IS the forensic report.

    70k tag fix: queries top-{MAX_TAGS_TO_LLM} most frequent tags by COUNT(*)
    instead of SELECT DISTINCT - prevents 429 / token overflow.
    The LLM selects vision_tag_filter from this frequency-ranked pool.
    """
    reference_image_part = None
    if reference_image_path:
        ref_mime = get_image_mime_type(reference_image_path)
        if reference_image_path.startswith("gs://"):
            reference_image_part = types.Part.from_uri(file_uri=reference_image_path, mime_type=ref_mime)
        else:
            with open(reference_image_path, "rb") as f:
                reference_image_part = types.Part.from_bytes(data=f.read(), mime_type=ref_mime)

    # Fetch top-N most frequent tags from DB (fixes 70k -> 429 overflow)
    # COUNT(*) frequency ordering = most semantically significant tags in the corpus.
    # The LLM picks vision_tag_filter from this pool to scope the tag search arm.
    available_tags = []
    try:
        engine, _ = await get_alloydb_connection()
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                (
                    f"SELECT tag, COUNT(*) AS freq "
                    f"FROM {DB_SCHEMA}.visual_assets, unnest(vision_tags) AS tag "
                    f"GROUP BY tag "
                    f"ORDER BY freq DESC "
                    f"LIMIT {MAX_TAGS_TO_LLM}"
                )
            )
            available_tags = [r["tag"] for r in rows if r["tag"]]
            print(f"   Loaded top-{len(available_tags)} most frequent tags from DB.")
    except Exception as e:
        print(f"   Tag query failed ({e}). Using static fallback ({len(WEB_AUDIT_VISION_TAGS)} tags).")

    print("1. Generating Audit Configuration (single Pro call: forensic + config)...")
    audit_config = generate_audit_config(user_goal, None, available_tags,
                                         reference_image_part=reference_image_part)
    if audit_config.get("image_description"):
        print(f"Forensic Analysis:\n{audit_config['image_description']}\n{'-'*50}")
    print("Config:\n", json.dumps(audit_config, indent=2))

    # For PERSON_SEARCH: run face detection once, store bytes for reuse
    search_mode = audit_config.get("search_mode", "GENERAL_ASSET")
    if search_mode == "PERSON_SEARCH" and reference_image_path:
        print("1b. Running Cloud Vision face detection (once)...")
        audit_config["_face_crop_bytes"] = extract_face_crop(reference_image_path)
    else:
        audit_config["_face_crop_bytes"] = None

    return audit_config


async def run_full_test_bench_pipeline_execution(audit_config: dict,
                                                  reference_image_path: Optional[str] = None,
                                                  quick_mode: bool = False) -> pd.DataFrame:
    """
    Stages 2-5: Retrieval -> Drop-off -> LLM Inference -> Summary.

    CPU reranker runs inside Stage 2 (zero latency, no LLM calls).
    Dedup (content_hash) reduces the candidate pool to unique images only.
    Smart Scan is capped at MAX_CANDIDATES_SMART_SCAN for bounded cost/latency.
    All stages are mode-aware.
    """
    try:
        from IPython.display import display, Markdown, HTML
    except ImportError:
        display = Markdown = HTML = None

    search_mode     = audit_config.get("search_mode", "GENERAL_ASSET")
    face_crop_bytes = audit_config.get("_face_crop_bytes")
    pipeline_start  = time.time()
    telemetry       = {}

    # Stage 2: Hybrid Search (returns unique images only, CPU reranked)
    t0 = time.time()
    print(f"\n2. Parallel 4-Arm Hybrid Search (Mode: {search_mode})...")
    search_results = await run_hybrid_search({}, audit_context=audit_config,
                                              reference_image_path=reference_image_path,
                                              face_crop_bytes=face_crop_bytes)
    telemetry["Stage 2 (4-Arm Search)"] = f"{time.time()-t0:.2f}s | {len(search_results)} unique candidates"
    print(f"   Unique candidates: {len(search_results)}")

    # EXACT_MATCH: flag near-pixel-perfect hits
    if search_mode == "EXACT_MATCH":
        exact_hits = [r for r in search_results if r.get("vector_distance", 1.0) < 0.05]
        if exact_hits:
            print(f"   Near-exact match(es) found: {len(exact_hits)} (vector_distance < 0.05)")

    # Stage 3: Drop-off Segmentation
    t0 = time.time()
    print(f"\n3. Drop-off Segmentation (Mode: {search_mode})...")
    df_results = pd.DataFrame(search_results)
    df_high, df_edge, df_low = detect_dropoff_flawless(df_results, search_mode=search_mode)
    telemetry["Stage 3 (Drop-off)"] = (
        f"{time.time()-t0:.2f}s | High:{len(df_high)} Edge:{len(df_edge)} Low:{len(df_low)}"
    )

    # Stage 3.5: REMOVED - CPU reranker already ran in Stage 2 (< 50ms, zero LLM calls)
    telemetry["Stage 3.5 (CPU Reranker)"] = "< 50ms | 0 LLM calls | Ran inside Stage 2"

    globals()["df_results_full"] = df_results

    if quick_mode:
        print(f"\n   Quick Mode: top {MAX_CANDIDATES_QUICK_MODE} per tier")
        candidates_high = df_high.head(MAX_CANDIDATES_QUICK_MODE)
        candidates_edge = df_edge.head(MAX_CANDIDATES_QUICK_MODE)
    else:
        # Bounded Smart Scan: High tier has priority, Edge fills the remaining budget.
        candidates_high = df_high.head(MAX_CANDIDATES_SMART_SCAN)
        edge_budget     = max(MAX_CANDIDATES_SMART_SCAN - len(candidates_high), 0)
        candidates_edge = df_edge.head(edge_budget)
        print(f"\n   Smart Scan: {len(candidates_high)} High + {len(candidates_edge)} Edge"
              f" (cap {MAX_CANDIDATES_SMART_SCAN})")

    # Stage 4: Parallel LLM Visual Inference
    # Each call: [IMAGE 0 (reference)] + [IMAGE 1 (candidate)] + [audit context text]
    t0 = time.time()
    print(f"\n4. Parallel LLM Audit Inference (Mode: {search_mode})...")
    results_df = await run_llm_inference_on_dropoff_results(
        candidates_high, candidates_edge, audit_config,
        reference_image_path=reference_image_path
    )
    inf_dur    = time.time() - t0
    throughput = len(results_df) / max(inf_dur, 0.01)
    telemetry["Stage 4 (LLM Inference)"] = f"{inf_dur:.2f}s | {throughput:.1f} img/s"

    # Stage 5: Summary
    t0 = time.time()
    print("\n5. Generating Calibration Summary...")
    summary = await generate_ai_audit_summary(results_df, audit_config)
    telemetry["Stage 5 (Summary)"] = f"{time.time()-t0:.2f}s"

    total_time  = time.time() - pipeline_start
    error_count = results_df["error"].notna().sum() if (not results_df.empty and "error" in results_df.columns) else 0
    high_conf   = (results_df["match_confidence"] >= 90).sum() if not results_df.empty else 0
    mid_conf    = ((results_df["match_confidence"] >= 60) & (results_df["match_confidence"] < 90)).sum() if not results_df.empty else 0
    low_conf    = (results_df["match_confidence"] < 60).sum() if not results_df.empty else 0
    total_pass  = results_df["matches_criteria"].sum() if not results_df.empty else 0

    telemetry_rows = "\n".join(
        f"| **{k}** | {v} |" for k, v in telemetry.items()
    )
    telemetry_md = (
        f"### Pipeline Telemetry - Mode: {search_mode}\n"
        f"| Stage | Result |\n|:---|:---|\n"
        f"| **Total Pipeline Time** | **{total_time:.2f}s** |\n"
        f"{telemetry_rows}\n"
        f"| **PASS / Total** | **{total_pass} / {len(results_df)}** |\n"
        f"| **Confidence >=90%** | {high_conf} |\n"
        f"| **Confidence 60-90%** | {mid_conf} |\n"
        f"| **Confidence <60%** | {low_conf} |\n"
        f"| **Errors** | {error_count} |\n"
    )

    if display and Markdown and HTML and not results_df.empty:
        display(Markdown(telemetry_md))
        display(Markdown(summary))
        display(Markdown("### Detailed Audit Results"))
        disp = results_df.copy()
        disp["Preview"] = disp["gcs_raw_path"].apply(
            lambda x: f'<img src="{get_gcs_image_base64(x)}" width="150" />' if x else "[No Preview]"
        )
        disp["Page"] = disp["page_url"].apply(
            lambda x: f'<a href="{x}" target="_blank">{x[:60]}</a>' if x else "[No Link]"
        )
        cols   = ["Preview", "matches_criteria", "relevance_score", "match_confidence",
                  "gcs_raw_path", "Page", "match_rationale"]
        labels = ["Preview", "Match", "Score", "Conf%", "GCS Path", "Page", "Rationale"]
        if "gemini_description" in disp.columns:
            cols.append("gemini_description"); labels.append("Description")
        display(HTML(disp[cols].rename(columns=dict(zip(cols, labels))).to_html(escape=False, index=False)))
    else:
        print("\n" + telemetry_md)
        print("\n" + summary)

    return results_df


async def run_full_test_bench_pipeline(user_goal: str,
                                        reference_image_path: Optional[str] = None,
                                        quick_mode: bool = False) -> pd.DataFrame:
    """Backwards-compatible wrapper. Runs the full Stage 1-5 pipeline."""
    config = await generate_audit_config_only(user_goal, reference_image_path)
    return await run_full_test_bench_pipeline_execution(config, reference_image_path, quick_mode=quick_mode)


## E2E Execution & Telemetry Helpers

This cell defines the core pipeline orchestrators: `generate_audit_config_only` (Stage 1 — single-call forensic analysis + config) and `run_full_test_bench_pipeline_execution` (Stage 2 E2E). It coordinates retrieval, segmentation, CPU reranking, and visual LLM inference, while logging telemetry and formatted scorecard widgets.

In [ ]:
# TEST RUN (Stage 1): Generate Audit Rules & Configuration
# Run this cell to upload your reference image and generate the rule configuration.
import nest_asyncio
nest_asyncio.apply()

# Dynamic Reference Image Detector (Triggered via Colab Interactive Upload)
reference_image_path = None
try:
    from google.colab import files
    print("[OPTIONAL] Upload a reference image for comparative compliance audit:")
    uploaded = files.upload()
    if uploaded:
        reference_image_path = list(uploaded.keys())[0]
        print(f" Reference image uploaded: {reference_image_path}")
    else:
        print(" No reference image uploaded. Running text-only audit goal.")
except Exception as e:
    reference_image_path = None

# Enterprise Audit Goal (Modify this as needed)
TEST_AUDIT_GOAL = "Find all pages with this exact image"

# Step 1: Generate configuration rules
audit_config_global = None
try:
    audit_config_global = await generate_audit_config_only(
        user_goal=TEST_AUDIT_GOAL,
        reference_image_path=reference_image_path
    )
    print("💡 TIP: You can inspect and tweak 'audit_config_global' directly in the cell below before running Stage 2.")
except Exception as e:
    print(f"Error generating audit configuration: {e}")


In [ ]:
# TEST RUN (Stage 2): Execute Visual Search & Parallel Audit
# Run this cell to execute retrieval, segmentation, and LLM inference.
# You can uncomment and modify rules below to calibrate config before executing.

if 'audit_config_global' in globals() and audit_config_global is not None:
    # OPTIONAL CALIBRATION TUNING:
    # If the generated AI rules were slightly off, you can uncomment and edit them here:
    # audit_config_global["inclusion_criteria"] = [
    #     "The image must contain the legacy Google Pay logo featuring the interlocking loops design."
    # ]
    # audit_config_global["exclusion_criteria"] = [
    #     "Exclude images containing the current compliant Google Pay GPay wordmark button logo."
    # ]

    df_results_global = None
    try:
        df_results_global = await run_full_test_bench_pipeline_execution(
            audit_config=audit_config_global,
            reference_image_path=reference_image_path
        )
    except Exception as e:
        print(f"Error executing test bench pipeline: {e}")
else:
    print(" Please run Stage 1 cell first to generate 'audit_config_global'.")
